# Ingest: one day from the orders API, plus the customer and product, into data + raw

In [1]:
import json
from pathlib import Path

In [2]:
%run source_api.ipynb

In [ ]:
# ROOT = Path.(__file__).parent
ROOT = Path.cwd().resolve()
RAW = ROOT / "data" / "raw"

In [9]:
print(ROOT, RAW)

D:\Learn Airflow D:\Learn Airflow\data\raw


In [12]:
import shutil

In [37]:
def ingest_day(day):
    out = RAW / "orders" / f"dt={day}"
    out.mkdir(parents=True, exist_ok=True)
    tmp = out / "orders.jsonl.tmp"
    
    page, count = 1, 0
    
    with open(tmp, "w") as f:
        while page:
            data = fetch_page(day, page)
            for order in data["orders"]:
                f.write(json.dumps(order) + "\n")
        
            count += len(data["orders"])
            page = data["next_page"]
    
    tmp.replace(out / "orders.jsonl")
    
    for name in ["customers.csv", "products.json"]:
        shutil.copy(ROOT / "source" / name, RAW / name)
    
    manifest = {"day": day, "orders": count, "path": f"data/raw/orders/dt={day}"}
    (out / "_manifest.json").write_text(json.dumps(manifest))
    return manifest

In [ ]:
# Test run
def sample_run(*argv):
    print(argv[0])
    day = argv[0]
    print(ingest_day(day))

sample_run("2026-11-01")

2026-11-01
{'day': '2026-11-01', 'orders': 6708, 'path': 'data/raw/orders/dt=2026-11-01'}
